In [ ]:
# ============================================================
# HRV Analysis & MS Monitoring Notebook
# Author: <Your Name>
# Purpose:
#   - Analyze HRV data using Python
#   - Interpret results through a Multiple Sclerosis (MS) lens
#   - Produce clinically cautious, reproducible insights
#
# NOTE:
#   This notebook is for monitoring & research purposes only.
#   It is NOT a diagnostic or clinical decision tool.
# ============================================================


# ============================================================
# 1. IMPORTS & ENVIRONMENT SETUP
# ============================================================

import pandas as pd
import numpy as np

from scipy import stats
from scipy.signal import welch

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime, timedelta

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Visualization defaults
plt.rcParams["figure.figsize"] = (12, 6)


# ============================================================
# 2. DOMAIN CONTEXT & ASSUMPTIONS (READ CAREFULLY)
# ============================================================

"""
HRV CONTEXT
-----------
- HRV reflects autonomic nervous system (ANS) balance
- RMSSD → Parasympathetic activity (most reliable for wearables)
- SDNN  → Overall variability (context-dependent)
- LF/HF → Controversial, interpret cautiously

MS CONTEXT
----------
- MS patients may show:
  - Reduced baseline HRV
  - Blunted recovery after stress
  - Higher sensitivity to fatigue, heat, infection
- HRV changes may reflect:
  - Autonomic dysfunction
  - Fatigue load
  - Pseudo-relapse conditions
  - Illness or medication effects

CLINICAL SAFETY
---------------
- HRV trends > single values
- Always contextualize with:
  - Sleep quality
  - Illness/infection
  - Heat exposure
  - Physical & cognitive load
"""


# ============================================================
# 3. DATA INGESTION
# ============================================================

"""
Expected data sources:
- RR intervals (preferred) OR
- Device-calculated HRV metrics (fallback)

Example inputs:
- Garmin CSV exports
- API-derived daily summaries
"""

# Example: Load RR interval data
# Replace with your actual file path
rr_file_path = "data/rr_intervals.csv"

# Expected columns:
# timestamp | rr_ms
try:
    rr_df = pd.read_csv(rr_file_path, parse_dates=["timestamp"])
except FileNotFoundError:
    rr_df = None
    print("RR interval file not found. Proceeding without RR-level data.")


# Example: Load daily HRV summary
daily_hrv_path = "data/daily_hrv.csv"

# Expected columns:
# date | rmssd | sdnn | resting_hr | sleep_score | body_battery
try:
    daily_df = pd.read_csv(daily_hrv_path, parse_dates=["date"])
except FileNotFoundError:
    daily_df = None
    print("Daily HRV file not found.")


# ============================================================
# 4. DATA QUALITY & PREPROCESSING
# ============================================================

def clean_rr_intervals(rr_df, min_rr=300, max_rr=2000):
    """
    Clean RR intervals:
    - Remove physiologically implausible values
    - Remove NaNs
    """
    rr = rr_df.copy()
    rr = rr.dropna(subset=["rr_ms"])
    rr = rr[(rr["rr_ms"] >= min_rr) & (rr["rr_ms"] <= max_rr)]
    return rr


if rr_df is not None:
    rr_df = clean_rr_intervals(rr_df)


# ============================================================
# 5. HRV METRIC CALCULATION
# ============================================================

def calculate_time_domain_hrv(rr_ms):
    """
    Calculate standard time-domain HRV metrics.
    """
    diff_rr = np.diff(rr_ms)

    metrics = {
        "mean_rr": np.mean(rr_ms),
        "sdnn": np.std(rr_ms, ddof=1),
        "rmssd": np.sqrt(np.mean(diff_rr**2)),
        "pnn50": np.sum(np.abs(diff_rr) > 50) / len(diff_rr) * 100
    }

    return metrics


def calculate_frequency_domain_hrv(rr_ms, fs=4.0):
    """
    Frequency domain HRV using Welch method.
    NOTE: Interpret LF/HF with caution in MS.
    """
    rr_sec = rr_ms / 1000.0
    rr_interp = np.interp(
        np.arange(0, len(rr_sec)),
        np.arange(0, len(rr_sec)),
        rr_sec
    )

    f, pxx = welch(rr_interp, fs=fs)

    lf_band = (0.04, 0.15)
    hf_band = (0.15, 0.40)

    lf_power = np.trapz(pxx[(f >= lf_band[0]) & (f <= lf_band[1])])
    hf_power = np.trapz(pxx[(f >= hf_band[0]) & (f <= hf_band[1])])

    return {
        "lf_power": lf_power,
        "hf_power": hf_power,
        "lf_hf_ratio": lf_power / hf_power if hf_power > 0 else np.nan
    }


if rr_df is not None:
    hrv_time = calculate_time_domain_hrv(rr_df["rr_ms"].values)
    hrv_freq = calculate_frequency_domain_hrv(rr_df["rr_ms"].values)
else:
    hrv_time, hrv_freq = None, None


# ============================================================
# 6. TREND & BASELINE ANALYSIS (MOST IMPORTANT FOR MS)
# ============================================================

def rolling_baseline(series, window=7):
    """
    Rolling baseline to detect meaningful deviations.
    """
    return series.rolling(window=window, min_periods=3).mean()


if daily_df is not None:
    daily_df = daily_df.sort_values("date")
    daily_df["rmssd_baseline"] = rolling_baseline(daily_df["rmssd"])
    daily_df["rmssd_delta"] = daily_df["rmssd"] - daily_df["rmssd_baseline"]


# ============================================================
# 7. VISUALIZATION
# ============================================================

if daily_df is not None:
    plt.plot(daily_df["date"], daily_df["rmssd"], label="RMSSD")
    plt.plot(daily_df["date"], daily_df["rmssd_baseline"], label="7-day Baseline")
    plt.axhline(daily_df["rmssd"].mean(), linestyle="--", alpha=0.5)
    plt.title("HRV (RMSSD) Trend – MS Monitoring Context")
    plt.xlabel("Date")
    plt.ylabel("RMSSD (ms)")
    plt.legend()
    plt.show()


# ============================================================
# 8. MS-SPECIFIC INTERPRETATION LOGIC
# ============================================================

def interpret_rmssd_change(delta):
    """
    MS-aware interpretation of RMSSD changes.
    """
    if delta < -10:
        return "Significant parasympathetic suppression – possible fatigue, illness, heat, or pseudo-relapse"
    elif delta < -5:
        return "Moderate reduction – monitor closely"
    elif delta > 5:
        return "Improved recovery / parasympathetic activation"
    else:
        return "Within normal variability"


if daily_df is not None:
    daily_df["ms_interpretation"] = daily_df["rmssd_delta"].apply(interpret_rmssd_change)


# ============================================================
# 9. ALTERNATIVE ANALYSES (OPTIONAL EXTENSIONS)
# ============================================================

"""
Optional extensions:
- Compare HRV vs sleep score
- HRV vs subjective fatigue
- Change-point detection
- Heat exposure correlation
- Relapse / pseudo-relapse annotation
"""


# ============================================================
# 10. PRACTICAL SUMMARY (AUTO-GENERATED)
# ============================================================

def generate_summary(df):
    latest = df.iloc[-1]
    summary = f"""
    HRV SUMMARY (Latest Day)
    ------------------------
    RMSSD: {latest['rmssd']:.1f} ms
    Baseline: {latest['rmssd_baseline']:.1f} ms
    Delta: {latest['rmssd_delta']:.1f} ms

    Interpretation:
    {latest['ms_interpretation']}

    Recommended Actions:
    - Check sleep quality
    - Review fatigue & heat exposure
    - Avoid overinterpreting single-day drops
    """
    return summary


if daily_df is not None:
    print(generate_summary(daily_df))

# ============================================================
# 11. RELAPSE / EVENT ANNOTATION
# ============================================================

"""
Annotation philosophy:
- User-provided or clinician-confirmed events
- NEVER inferred automatically
- Used only for interpretation and visualization
"""

# Example annotation table
annotations = pd.DataFrame({
    "date": [
        "2025-01-10",
        "2025-02-03"
    ],
    "event_type": [
        "Pseudo-relapse (heat)",
        "Confirmed relapse"
    ],
    "notes": [
        "Heatwave + fatigue",
        "Neurologist-confirmed relapse"
    ]
})

annotations["date"] = pd.to_datetime(annotations["date"])

if daily_df is not None:
    daily_df = daily_df.merge(
        annotations,
        on="date",
        how="left"
    )


# ============================================================
# 12. STATISTICAL ALERTING LOGIC
# ============================================================

def compute_z_score(series, window=30):
    rolling_mean = series.rolling(window, min_periods=10).mean()
    rolling_std = series.rolling(window, min_periods=10).std()
    return (series - rolling_mean) / rolling_std


if daily_df is not None:
    daily_df["rmssd_z"] = compute_z_score(daily_df["rmssd"])

    def alert_logic(z):
        if z <= -2.0:
            return "CRITICAL DROP"
        elif z <= -1.5:
            return "WARNING DROP"
        elif z >= 1.5:
            return "POSITIVE RECOVERY"
        else:
            return "NORMAL"

    daily_df["alert_level"] = daily_df["rmssd_z"].apply(alert_logic)

# ============================================================
# 12. STATISTICAL ALERTING LOGIC
# ============================================================

def compute_z_score(series, window=30):
    rolling_mean = series.rolling(window, min_periods=10).mean()
    rolling_std = series.rolling(window, min_periods=10).std()
    return (series - rolling_mean) / rolling_std


if daily_df is not None:
    daily_df["rmssd_z"] = compute_z_score(daily_df["rmssd"])

    def alert_logic(z):
        if z <= -2.0:
            return "CRITICAL DROP"
        elif z <= -1.5:
            return "WARNING DROP"
        elif z >= 1.5:
            return "POSITIVE RECOVERY"
        else:
            return "NORMAL"

    daily_df["alert_level"] = daily_df["rmssd_z"].apply(alert_logic)


# ============================================================
# 12. STATISTICAL ALERTING LOGIC
# ============================================================

def compute_z_score(series, window=30):
    rolling_mean = series.rolling(window, min_periods=10).mean()
    rolling_std = series.rolling(window, min_periods=10).std()
    return (series - rolling_mean) / rolling_std


if daily_df is not None:
    daily_df["rmssd_z"] = compute_z_score(daily_df["rmssd"])

    def alert_logic(z):
        if z <= -2.0:
            return "CRITICAL DROP"
        elif z <= -1.5:
            return "WARNING DROP"
        elif z >= 1.5:
            return "POSITIVE RECOVERY"
        else:
            return "NORMAL"

    daily_df["alert_level"] = daily_df["rmssd_z"].apply(alert_logic)

# ============================================================
# 13. AUTOMATED WEEKLY REPORT
# ============================================================

def generate_weekly_report(df):
    last_7 = df.tail(7)

    report = {
        "avg_rmssd": last_7["rmssd"].mean(),
        "rmssd_trend": last_7["rmssd"].iloc[-1] - last_7["rmssd"].iloc[0],
        "alerts": last_7["alert_level"].value_counts().to_dict(),
        "events": last_7["event_type"].dropna().tolist()
    }

    narrative = f"""
    WEEKLY HRV REPORT
    =================
    Average RMSSD: {report['avg_rmssd']:.1f} ms
    Weekly Trend: {report['rmssd_trend']:.1f} ms

    Alerts:
    {report['alerts']}

    Annotated Events:
    {report['events'] if report['events'] else 'None'}

    Interpretation:
    - Focus on trend, not daily noise
    - Review sleep, fatigue, and heat exposure
    - Escalate only if sustained drops persist >7–10 days
    """

    return narrative


if daily_df is not None:
    print(generate_weekly_report(daily_df))



# ============================================================
# 14. PLOTLY DASHBOARD
# ============================================================

import plotly.express as px

if daily_df is not None:
    fig = px.line(
        daily_df,
        x="date",
        y="rmssd",
        color="alert_level",
        title="HRV RMSSD Trend with Alerts (MS Context)",
        markers=True
    )

    fig.add_scatter(
        x=daily_df["date"],
        y=daily_df["rmssd_baseline"],
        mode="lines",
        name="Baseline",
        line=dict(dash="dash")
    )

    fig.show()




# ============================================================
# 15. EXPORT FOR POWER BI
# ============================================================

if daily_df is not None:
    daily_df.to_csv("output/hrv_ms_monitoring_dataset.csv", index=False)


# ============================================================
# 16. CHANGE-POINT DETECTION
# ============================================================

"""
Goal:
Detect sustained shifts in HRV baseline.

MS context:
- A change point is NOT a relapse diagnosis.
- It may indicate altered autonomic state, illness, heat stress,
  medication change, poor sleep, fatigue accumulation, or recovery shift.
"""

# Optional install:
# pip install ruptures

import numpy as np
import pandas as pd

try:
    import ruptures as rpt
    RUPTURES_AVAILABLE = True
except ImportError:
    RUPTURES_AVAILABLE = False


def detect_change_points(df, column="rmssd", penalty=5):
    """
    Detect change points in a daily HRV signal.
    Uses PELT if ruptures is installed.
    Falls back to rolling z-score flags otherwise.
    """
    data = df[column].dropna().values

    if len(data) < 14:
        df["change_point"] = False
        return df

    df = df.copy()
    df["change_point"] = False

    if RUPTURES_AVAILABLE:
        model = rpt.Pelt(model="rbf").fit(data)
        change_indices = model.predict(pen=penalty)

        valid_indices = [idx for idx in change_indices if idx < len(df)]

        for idx in valid_indices:
            df.loc[df.index[idx], "change_point"] = True

    else:
        rolling_mean = df[column].rolling(14, min_periods=7).mean()
        rolling_std = df[column].rolling(14, min_periods=7).std()
        z = (df[column] - rolling_mean) / rolling_std
        df["change_point"] = z.abs() >= 2.0

    return df


if daily_df is not None:
    daily_df = detect_change_points(daily_df, column="rmssd")

# ============================================================
# 17. FATIGUE RISK MODELING — NON-DIAGNOSTIC
# ============================================================

"""
Goal:
Estimate probability of high fatigue risk.

Important:
- This is a monitoring aid, not a medical model.
- Requires self-reported fatigue labels.
- Recommended scale: 0–10 daily fatigue score.
"""

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score


def prepare_fatigue_features(df):
    df = df.copy()

    df["rmssd_7d_mean"] = df["rmssd"].rolling(7, min_periods=3).mean()
    df["rmssd_7d_std"] = df["rmssd"].rolling(7, min_periods=3).std()
    df["rmssd_pct_change"] = df["rmssd"].pct_change()

    if "resting_hr" in df.columns:
        df["rhr_7d_mean"] = df["resting_hr"].rolling(7, min_periods=3).mean()

    if "sleep_score" in df.columns:
        df["sleep_7d_mean"] = df["sleep_score"].rolling(7, min_periods=3).mean()

    if "body_battery" in df.columns:
        df["body_battery_7d_mean"] = df["body_battery"].rolling(7, min_periods=3).mean()

    return df


def train_fatigue_model(df, fatigue_column="fatigue_score"):
    """
    fatigue_score expected: 0–10
    High fatigue label: >= 7
    """

    df = prepare_fatigue_features(df)

    df["high_fatigue"] = (df[fatigue_column] >= 7).astype(int)

    candidate_features = [
        "rmssd",
        "rmssd_baseline",
        "rmssd_delta",
        "rmssd_z",
        "rmssd_7d_mean",
        "rmssd_7d_std",
        "rmssd_pct_change",
        "resting_hr",
        "rhr_7d_mean",
        "sleep_score",
        "sleep_7d_mean",
        "body_battery",
        "body_battery_7d_mean"
    ]

    features = [c for c in candidate_features if c in df.columns]

    model_df = df.dropna(subset=features + ["high_fatigue"])

    X = model_df[features]
    y = model_df["high_fatigue"]

    if len(model_df) < 30:
        raise ValueError("Need at least ~30 labeled days for a minimally useful fatigue model.")

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000))
    ])

    pipeline.fit(X, y)

    model_df["fatigue_risk_probability"] = pipeline.predict_proba(X)[:, 1]

    return pipeline, model_df, features


if daily_df is not None and "fatigue_score" in daily_df.columns:
    fatigue_model, fatigue_model_df, fatigue_features = train_fatigue_model(daily_df)

# ============================================================
# 18. GARMIN INGESTION
# ============================================================

"""
Two paths:

1. Official Garmin Health API
   - Best for research / clinical / production use
   - Requires Garmin approval
   - Server-side, partner-oriented

2. Unofficial garminconnect package
   - Useful for personal analysis
   - Not guaranteed stable
   - Do not use for regulated clinical workflows
"""

# Optional install:
# pip install garminconnect python-dotenv

import os
from datetime import date, timedelta


def ingest_garmin_personal(start_date, end_date):
    """
    Personal Garmin Connect ingestion using unofficial garminconnect library.

    Requires:
    GARMIN_EMAIL
    GARMIN_PASSWORD

    Returns:
    daily_df-like dataframe
    """

    from garminconnect import Garmin

    email = os.getenv("GARMIN_EMAIL")
    password = os.getenv("GARMIN_PASSWORD")

    if not email or not password:
        raise EnvironmentError("Set GARMIN_EMAIL and GARMIN_PASSWORD as environment variables.")

    client = Garmin(email, password)
    client.login()

    rows = []

    current = pd.to_datetime(start_date).date()
    end = pd.to_datetime(end_date).date()

    while current <= end:
        current_str = current.isoformat()

        try:
            stats = client.get_stats(current_str)
            sleep = client.get_sleep_data(current_str)

            rows.append({
                "date": current_str,
                "resting_hr": stats.get("restingHeartRate"),
                "steps": stats.get("totalSteps"),
                "stress_avg": stats.get("averageStressLevel"),
                "body_battery": stats.get("bodyBatteryMostRecentValue"),
                "sleep_score": sleep.get("dailySleepDTO", {}).get("sleepScores", {}).get("overall", {}).get("value")
            })

        except Exception as e:
            rows.append({
                "date": current_str,
                "error": str(e)
            })

        current += timedelta(days=1)

    garmin_df = pd.DataFrame(rows)
    garmin_df["date"] = pd.to_datetime(garmin_df["date"])

    return garmin_df


# Example:
# garmin_df = ingest_garmin_personal("2026-01-01", "2026-01-31")
# daily_df = daily_df.merge(garmin_df, on="date", how="left")

# ============================================================
# 19. CLINICAL-GRADE PDF REPORT
# ============================================================

"""
Goal:
Generate a structured PDF for personal tracking or clinician discussion.

Important:
- The report must avoid diagnostic language.
- Use "monitoring signal", "trend", "association", "requires clinical context".
"""

# Optional install:
# pip install reportlab

from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
)
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors


def create_hrv_plot_for_pdf(df, output_path="output/hrv_trend.png"):
    import matplotlib.pyplot as plt
    import os

    os.makedirs("output", exist_ok=True)

    plt.figure(figsize=(10, 5))
    plt.plot(df["date"], df["rmssd"], label="RMSSD")
    plt.plot(df["date"], df["rmssd_baseline"], linestyle="--", label="Baseline")

    if "change_point" in df.columns:
        cp = df[df["change_point"] == True]
        plt.scatter(cp["date"], cp["rmssd"], label="Change Point")

    plt.title("RMSSD Trend with Baseline")
    plt.xlabel("Date")
    plt.ylabel("RMSSD ms")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()

    return output_path


def generate_clinical_pdf_report(
    df,
    output_pdf="output/hrv_ms_clinical_report.pdf",
    patient_label="Patient",
    report_period="Last 7 days"
):
    import os

    os.makedirs("output", exist_ok=True)

    styles = getSampleStyleSheet()
    doc = SimpleDocTemplate(output_pdf, pagesize=A4)

    story = []

    latest = df.iloc[-1]
    last_7 = df.tail(7)

    story.append(Paragraph("HRV & MS Monitoring Report", styles["Title"]))
    story.append(Spacer(1, 12))

    story.append(Paragraph(f"<b>Patient:</b> {patient_label}", styles["Normal"]))
    story.append(Paragraph(f"<b>Report period:</b> {report_period}", styles["Normal"]))
    story.append(Paragraph(f"<b>Generated:</b> {pd.Timestamp.today().date()}", styles["Normal"]))
    story.append(Spacer(1, 12))

    disclaimer = """
    This report is for monitoring and clinician discussion only.
    It does not diagnose relapse, disease activity, treatment failure,
    autonomic dysfunction, or any medical condition.
    """
    story.append(Paragraph(disclaimer, styles["Italic"]))
    story.append(Spacer(1, 12))

    summary_data = [
        ["Metric", "Value"],
        ["Latest RMSSD", f"{latest.get('rmssd', np.nan):.1f} ms"],
        ["Latest Baseline", f"{latest.get('rmssd_baseline', np.nan):.1f} ms"],
        ["Latest Delta", f"{latest.get('rmssd_delta', np.nan):.1f} ms"],
        ["Latest Alert", str(latest.get("alert_level", "N/A"))],
        ["7-day Average RMSSD", f"{last_7['rmssd'].mean():.1f} ms"],
        ["Change Points in Period", str(int(last_7.get("change_point", pd.Series(False)).sum()))]
    ]

    table = Table(summary_data)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("PADDING", (0, 0), (-1, -1), 6),
    ]))

    story.append(Paragraph("Executive Summary", styles["Heading2"]))
    story.append(table)
    story.append(Spacer(1, 16))

    plot_path = create_hrv_plot_for_pdf(df)
    story.append(Paragraph("HRV Trend", styles["Heading2"]))
    story.append(Image(plot_path, width=480, height=240))
    story.append(Spacer(1, 16))

    interpretation = f"""
    Latest interpretation: {latest.get("ms_interpretation", "N/A")}.
    Alert state: {latest.get("alert_level", "N/A")}.
    Any sustained HRV suppression should be interpreted alongside sleep,
    infection signs, heat exposure, medication changes, fatigue, and neurological symptoms.
    """
    story.append(Paragraph("Clinical Interpretation Notes", styles["Heading2"]))
    story.append(Paragraph(interpretation, styles["Normal"]))
    story.append(Spacer(1, 12))

    action_plan = """
    Recommended next steps:
    1. Review sleep, illness, heat exposure, medication changes, and fatigue.
    2. Track whether HRV suppression persists for 7–10 days.
    3. Annotate any neurological symptom changes separately.
    4. Contact a clinician if new or worsening neurological symptoms persist.
    """
    story.append(Paragraph("Practical Action Plan", styles["Heading2"]))
    story.append(Paragraph(action_plan, styles["Normal"]))

    doc.build(story)

    return output_pdf


# Example:
# pdf_path = generate_clinical_pdf_report(daily_df, patient_label="Stelios")
# print(pdf_path)

# ============================================================
# END OF NOTEBOOK
# ============================================================